# AI Agent Week 2: Transformer Cont.

04-11-2026

HF Transformers Course Implementation

## Last Week's Recap
* LLM Model structure: self attention (Q, K, V), multi-head, positional encoding,  (MoE) Feed-Forward Network (FFN). 
* Three architecture families: Encoder (BERT), Decoder (GPT), Encoder-Decoder (T5) and why they suits for different tasks.
* LLM model alone suitable for text/coding generation. But need Tools, external knowledge, and memory to be more useful in real world applications.
* Minimal agent loop: receive goal, build context, generate action, tool execution, receive feedback, repeat until goal is achieved.

# This Week: HF Transformers Course Ch1-3 and LLM Continuation

Reference: 
- [HF LLM Course](https://huggingface.co/docs/transformers), 
- [HF Transformer](https://huggingface.co/docs/transformers)


**HF Pipeline Tasks**
- Tasks v.s. Models
- BERT Model 

**Tokenization**
- Padding and truncation
- Special tokens
- Mask: Causal, Padding. 
- Embeddings

**Inference Optimization**
- KV Cache
- Mixed Precision / Quantization

**Fine Tuning**
- Mixed precision
- Training arguments
- Learning Curve
- Accelerater: Distributed training

**RL Post Training**

In [1]:
!pip install datasets evaluate transformers[sentencepiece] huggingface-hub tqdm

import numpy as np
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from huggingface_hub import login, notebook_login
from datasets import load_dataset
import evaluate


import torch.nn.functional as F
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    get_scheduler,
    BertTokenizer,
)
 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


## 1. Pipelines and Tasks
you need different models for different tasks. Models are fine-tuned for specific tasks and can't be swapped  interchangeably

In [5]:
# classification example. label sentence with user defined classes
from transformers import pipeline

classifier = pipeline("zero-shot-classification"  ,model="facebook/bart-large-mnli")
 
classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ],
    candidate_labels=["abjection", "acceptance"],
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[{'sequence': "I've been waiting for a HuggingFace course my whole life.",
  'labels': ['acceptance', 'abjection'],
  'scores': [0.875693678855896, 0.12430630624294281]},
 {'sequence': 'I hate this so much!',
  'labels': ['abjection', 'acceptance'],
  'scores': [0.9025610089302063, 0.09743904322385788]}]

In [6]:
# generate text example. 
generator = pipeline("text-generation", model="openai-community/gpt2", revision="607a30d")
generator("In order to be healthy, you should", max_new_tokens=20, num_return_sequences=1)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'In order to be healthy, you should be able to eat well and be in good shape. The body that produces the food you need to'}]

In [7]:
from transformers import GenerationConfig

generator = pipeline("text-generation", model="distilgpt2")
generator_config = GenerationConfig(max_new_tokens=20, num_return_sequences=1)
generator("In order to be healthy, you should", generation_config=generator_config) 

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'In order to be healthy, you should be able to eat a lot of vegetables.\n\n\n\n\n\n\n\n\n\n\n'}]

In [8]:
# fill mask task. also use top_k
unmasker = pipeline("fill-mask") # default model is "distilbert/distilroberta-base"
unmasker("To keep healthy, you must eat <mask> food.", top_k=2)

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[{'score': 0.3501476049423218,
  'token': 2245,
  'token_str': ' healthy',
  'sequence': 'To keep healthy, you must eat healthy food.'},
 {'score': 0.3332260251045227,
  'token': 30426,
  'token_str': ' nutritious',
  'sequence': 'To keep healthy, you must eat nutritious food.'}]

In [10]:
 

question_answerer = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")
question_answerer(
    question="What type of chocolate do I like?", 
    context="I like dark chocolate",
)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

{'score': 0.5632748007774353,
 'start': 7,
 'end': 21,
 'answer': 'dark chocolate'}

**An example of LLM model limitation: only generative model have the reasoning ability. BERT can only get information from context.**

In [9]:
from transformers import pipeline

question_answerer = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")
question_answerer(
    question="What type of chocolate do I like?", 
    context="I do not like to eat sugar or milk",
)


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'score': 0.5453944206237793,
 'start': 21,
 'end': 34,
 'answer': 'sugar or milk'}

**the result is not working**

BERT has no reasoning ability. It cannot:                                                                                                                                                                          
  - Infer what you do like from what you don't like                                                                                                                                                                
  - Generate words not present in the context                                                                                                                                                                      
  - Understand negation ("do not like") logically

In [11]:

qa = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")               

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [12]:
qa("Question:  I do not like sugar nor milk. What type of chocolate do I like?", max_new_tokens=20)  

Both `max_new_tokens` (=20) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Question:  I do not like sugar nor milk. What type of chocolate do I like?\nAnswer:  I like dark chocolate. It is not only healthy, but it'}]

## Model Architecture & Task Selection

###
- Encoder: BERT. Fill mask. 
- Decoder: GPT. Generate text.
- Encoder+Decoder: BART (Bidirectional and Auto-Regressive Transformers) for text summary.

### One model does not fit all tasks — each is fine-tuned with a task-specific output head.

| Task | Pipeline | Example |
|---|---|---|
| Text generation | `text-generation` | `openai-community/gpt2` |
| Sentiment analysis | `text-classification` | `distilbert-base-uncased-finetuned-sst-2-english` |
| Zero-shot classification | `zero-shot-classification` | `facebook/bart-large-mnli` |
| Fill mask | `fill-mask` | `distilbert-base-uncased` |
| Summarization | `summarization` | `sshleifer/distilbart-cnn-12-6` |
| Question answering | `question-answering` | `distilbert-base-cased-distilled-squad` |
| NER | `ner` | `dbmdz/bert-large-cased-finetuned-conll03-english` |


**Exception:** Instruction-tuned models (e.g. `meta-llama/Llama-3.2-3B-Instruct`) handle many tasks via prompting — still through `text-generation`.

---

### Three Architecture Families

| Architecture | Attention | Pretraining | Best For | Examples |
|---|---|---|---|---|
| **Encoder-only** | Bidirectional | MLM | Classification, NER, QA | BERT, DistilBERT, ModernBERT |
| **Decoder-only** | Causal | CLM | Generation, reasoning | GPT-2, LLaMA, Mistral |
| **Encoder-decoder** | Both | Seq2seq | Translation, summarization | BART, T5, FLAN-T5 |

- **MLM:** Predicts masked tokens using both left + right context → bidirectional. *(BERT)*
- **CLM:** Predicts next token from left context only → autoregressive. *(GPT)*

---

### Attention Internals
 
/
For encoder-decoder model, 

```
Encoder:  Input → [Bidirectional Self-Attention] → FFN → Contextual representation
Decoder:  Output → [Causal Masked Self-Attention] → [Cross-Attention ← KV from Encoder] → FFN → Next token
```

| Location | Attention Type | Q | K, V |
|---|---|---|---|
| Encoder | Self-attention | Encoder input | Encoder input |
| Decoder (1) | Masked self-attention | Decoder input | Decoder input |
| Decoder (2) | Cross-attention | Decoder | Encoder |


/

 

![The architecture of encoder-decoder models](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter1/transformers_architecture.png)

#### Bert Model Structure


BERT is an encoder-only model and is the first model to effectively implement deep bidirectionality to learn richer representations of the text by attending to words on both sides.

Below is Bert model for sequence classification. The input is two sentences. The output is whether those are from the same corpus. More see fine tune section below. https://huggingface.co/docs/transformers/tasks/sequence_classification
 

The model included building blocks as
- Embedding, positional encoding
- Self attention
- Layer norm
- FFN
- Dense MLP
- Classifier

```
BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (intermediate): BertIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermediate_act_fn): GELUActivation()
          )
          (output): BertOutput(
            (dense): Linear(in_features=3072, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
    )
    (pooler): BertPooler(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (activation): Tanh()
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=768, out_features=2, bias=True)
)
```

![Decoder Only Model GPT-2](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/gpt2_architecture.png)

# 2. Tokenizer

## Tokenizer

**Encoding pipeline:** Text → Tokens → IDs (numerical representations the model uses for computation)

| Concept | Description |
|---|---|
| **Token** | Human-readable text piece (word, subword, or character) |
| **ID** | Integer index of a token in the model's vocabulary |
| **Embedding** | Dense continuous vector mapped from a token ID — captures semantic/syntactic relationships |

**Batch processing requires uniform shape** (tensors are rectangular), handled by:

| Mechanism | Purpose |
|---|---|
| **Padding** | Adds `[PAD]` tokens to shorter sequences to match the longest in the batch |
| **Truncation** | Cuts sequences exceeding the model's max length |
| **Attention Mask** | Binary mask — `1` = attend, `0` = ignore (marks PAD tokens) |

#### Different model and tokenizer checkpoint will lead to different tokenization results.

In [13]:
text_to_tokenize = "Jim Henson was a prompt engineer. He want to be an AI engineer."
tokenized_text = text_to_tokenize.split()
print('tokenized result:', tokenized_text)

tokenizer1 = AutoTokenizer.from_pretrained("bert-base-cased")
tokenizer2 = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")



tokenized result: ['Jim', 'Henson', 'was', 'a', 'prompt', 'engineer.', 'He', 'want', 'to', 'be', 'an', 'AI', 'engineer.']


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [3]:

token1 = tokenizer1(text_to_tokenize)
token1b = tokenizer1.tokenize(text_to_tokenize)
token2 = tokenizer2(text_to_tokenize)
token2b = tokenizer2.tokenize(text_to_tokenize)

print('tokenizer1 tokenize result:', token1b)
print('tokenizer1 result:', token1)
print('tokenizer2 tokenize result:', token2b) 
print('tokenizer2 result:', token2)

tokenizer1 tokenize result: ['Jim', 'He', '##nson', 'was', 'a', 'pro', '##mpt', 'engineer', '.', 'He', 'want', 'to', 'be', 'an', 'AI', 'engineer', '.']
tokenizer1 result: {'input_ids': [101, 3104, 1124, 15703, 1108, 170, 5250, 18378, 3806, 119, 1124, 1328, 1106, 1129, 1126, 19016, 3806, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
tokenizer2 tokenize result: ['jim', 'henson', 'was', 'a', 'prompt', 'engineer', '.', 'he', 'want', 'to', 'be', 'an', 'ai', 'engineer', '.']
tokenizer2 result: {'input_ids': [101, 3958, 27227, 2001, 1037, 25732, 3992, 1012, 2002, 2215, 2000, 2022, 2019, 9932, 3992, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


AutoTokenizer is just a factory/router. When you call AutoTokenizer.from_pretrained("bert-base-cased"), it reads the model config, sees the model type is bert, and internally instantiates a BertTokenizer. They end up being the same object class.

In [23]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-cased")
tokenizer(text_to_tokenize)

{'input_ids': [101, 3104, 1124, 15703, 1108, 170, 5250, 18378, 3806, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:

from transformers import AutoTokenizer, BertTokenizer

t1 = AutoTokenizer.from_pretrained("bert-base-cased")
t2 = BertTokenizer.from_pretrained("bert-base-cased")
                                                                                                                                                                                                                
type(t1)  # → <class 'transformers.models.bert.tokenization_bert.BertTokenizer'>                                                                                                                                 , type(t2)  # → <class 'transformers.models.bert.tokenization_bert.BertTokenizer'>      

transformers.models.bert.tokenization_bert.BertTokenizer

In [25]:
type(t2) 

transformers.models.bert.tokenization_bert.BertTokenizer

In [ ]:
# with decoding, we can convert the token ids back to the original string.
decoded_string = tokenizer1.decode([101, 3104, 1124, 15703, 1108, 170, 5250, 18378, 3806, 119, 1124, 1328, 1106, 1129, 1126, 19016, 3806, 119, 102])
print(decoded_string)

[CLS] Jim Henson was a prompt engineer. He want to be an AI engineer. [SEP]


#### Padding 

In [5]:
# special tokens 
tokenizer1.cls_token_id, tokenizer1.pad_token_id, tokenizer1.sep_token_id, tokenizer1.mask_token_id

(101, 0, 102, 103)

In [6]:
tokenizer2.cls_token_id, tokenizer2.pad_token_id, tokenizer2.sep_token_id, tokenizer2.mask_token_id

(101, 0, 102, 103)

- Model and tokenizer must come from the same checkpoint. Every Model Was Trained With One Specific TokenizerDuring training, the model learned to associate token IDs with meanings. Token ID 4732 might mean " Paris" in one tokenizer, but mean something completely different in another tokenizer's vocabulary.GPT-2 tokenizer:    "I love Paris"  →  [40, 1842, 6342]
BERT tokenizer:     "I love Paris"  →  [1045, 2293, 3000]Same words, completely different numbers. The model only ever saw one of these mappings during training.

- Each sequence need to be of the same length, padded to the length of the longest sequence in the batch (dynamic padding)
- when batching padded sequences you must pass an attention mask so the model ignores PAD tokens; with proper padding + attention mask, batched and separate inputs produce the same results.


In [9]:


checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
#
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
sequence1_ids = [[100, 200, 200]]
sequence2_ids = [[100, 200]]

batched_ids = [
    [100, 200, 200],
    [100, 200, tokenizer.pad_token_id],
]

print("sequence separately:")
print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)

print("batched:")
print(model(torch.tensor(batched_ids)).logits)
print("individual sequences in batch:")
print(model(torch.tensor([batched_ids[0]])).logits)
print(model(torch.tensor([batched_ids[1]])).logits)


attention_mask = [
    [1, 1, 1],
    [1, 1, 0],
]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))


sequence separately:


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


tensor([[ 0.6514, -0.5004]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5223, -0.3537]], grad_fn=<AddmmBackward0>)
batched:
tensor([[ 0.6514, -0.5004],
        [ 0.6465, -0.5023]], grad_fn=<AddmmBackward0>)
individual sequences in batch:
tensor([[ 0.6514, -0.5004]], grad_fn=<AddmmBackward0>)
tensor([[ 0.6465, -0.5023]], grad_fn=<AddmmBackward0>)
with attention mask: tensor([[ 0.6514, -0.5004],
        [ 0.5223, -0.3537]], grad_fn=<AddmmBackward0>)


In [10]:
print("with attention mask:\n", outputs.logits)

with attention mask:
 tensor([[ 0.6514, -0.5004],
        [ 0.5223, -0.3537]], grad_fn=<AddmmBackward0>)


* Different types of padding:
  - **Dynamic padding:** Pad to the longest sequence in the batch (more efficient)
  - **Static padding:** Pad to a fixed max length (simpler but can waste compute)
* token_type_ids: segment embedding to distinguish different sentences in tasks like next sentence prediction (BERT) — not needed for single-sentence tasks or decoder-only models.

In [ ]:
# Will pad the sequences up to the maximum sequence length
model_inputs = tokenizer(text_to_tokenize, padding="longest")
print(f"print {model_inputs}")
# Will pad the sequences up to the model max length
# (512 for BERT or DistilBERT)
model_inputs = tokenizer(text_to_tokenize, padding="max_length")
print(f"print {model_inputs}")

# Will pad the sequences up to the specified max length
model_inputs = tokenizer(text_to_tokenize, padding="max_length", max_length=8)
print(f"print {model_inputs}")
 

print {'input_ids': [101, 3958, 27227, 2001, 1037, 25732, 3992, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}
print {'input_ids': [101, 3958, 27227, 2001, 1037, 25732, 3992, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [16]:
# max_length is configurable in the tokenizer, and it is 512 for BERT and DistilBERT. If the input sequence is longer than the max length, it will be truncated to fit the model's input size. The truncation can be done in different ways, such as truncating from the left, right, or both sides of the sequence. By default, the truncation is done from the right side of the sequence.
sequences = [                                                                                                                                                                                                    
      ("I've been waiting for a HuggingFace course my whole life.", "So have I!"),                                                                                                                                 
      ("What is your name?", "My name is Sylvain. Sorry can't hear you."),  
      ("Jim Henson was a prompt engineer.", "He want to be an AI engineer."),                                                                                                                                                        
  ]  
model_inputs = tokenizer(sequences, truncation=True)
print(f"model_inputs: {model_inputs}")

# Will truncate the sequences that are longer than the specified max length
model_inputs = tokenizer(sequences, max_length=8, truncation=True)
print(f"model_inputs: {model_inputs}")


model_inputs: {'input_ids': [[101, 1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012, 102, 2061, 2031, 1045, 999, 102], [101, 2054, 2003, 2115, 2171, 1029, 102, 2026, 2171, 2003, 25353, 22144, 2378, 1012, 3374, 2064, 1005, 1056, 2963, 2017, 1012, 102], [101, 3958, 27227, 2001, 1037, 25732, 3992, 1012, 102, 2002, 2215, 2000, 2022, 2019, 9932, 3992, 1012, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}
model_inputs: {'input_ids': [[101, 1045, 1005, 2310, 102, 2061, 2031, 102], [101, 2054, 2003, 102, 2026, 2171, 2003, 102], [101, 3958, 27227, 102, 2002, 2215, 2000, 102]], 'token_type_

In [50]:
tokenizer.decode(tokenizer.pad_token_id)

'[PAD]'

In [52]:
tokenizer.decode(tokenizer.cls_token_id)

'[CLS]'

**Convert output logits to probabilities using softmax, and then to predicted class labels using argmax:**

In [27]:
# checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
# model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
# tokenizer = AutoTokenizer.from_pretrained(checkpoint)
text_to_tokenize = "Jim Henson was a prompt engineer. He want to be an AI engineer."

tokens = tokenizer.tokenize(text_to_tokenize)
print("Tokens:", tokens)
ids = tokenizer.convert_tokens_to_ids(tokens)
print("Token IDs:", ids)
input_ids = torch.tensor([ids])
print("Input IDs:", input_ids) 

output = model(input_ids)
print("Logits:", output.logits)
 
import torch.nn.functional as F   
probs = F.softmax(output.logits, dim=-1)
print(probs)   


Tokens: ['jim', 'henson', 'was', 'a', 'prompt', 'engineer', '.', 'he', 'want', 'to', 'be', 'an', 'ai', 'engineer', '.']
Token IDs: [3958, 27227, 2001, 1037, 25732, 3992, 1012, 2002, 2215, 2000, 2022, 2019, 9932, 3992, 1012]
Input IDs: tensor([[ 3958, 27227,  2001,  1037, 25732,  3992,  1012,  2002,  2215,  2000,
          2022,  2019,  9932,  3992,  1012]])
Logits: tensor([[ 2.2928, -2.0055]], grad_fn=<AddmmBackward0>)
tensor([[0.9866, 0.0134]], grad_fn=<SoftmaxBackward0>)


In [46]:
# Sentiment Analysis with DistilBERT

sequences = [                                                                                                                                                                                                    
      ("I've been waiting for a HuggingFace course my whole life.", "So have I!"),   
      ("What is your name?", "My name is Sam!"),  
      ("Jim Henson was a prompt engineer.", "He wants to be an AI engineer!"),                                                                                                                                  
      ("Jim Henson was a prompt engineer.", "He wants to be an AI engineer."),                                                                                                                                                          
  ]  


ids = tokenizer(sequences, truncation=True, padding=True, return_tensors="pt") 
 
print("ids:", ids)

output = model(**ids)
print("Logits:", output.logits)

print("Use softmax to convert logits to probabilities:")
probs = F.softmax(output.logits, dim=-1)
print("Probs:",probs)   


ids: {'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102,  2061,  2031,  1045,   999,
           102],
        [  101,  2054,  2003,  2115,  2171,  1029,   102,  2026,  2171,  2003,
          3520,   999,   102,     0,     0,     0,     0,     0,     0,     0,
             0],
        [  101,  3958, 27227,  2001,  1037, 25732,  3992,  1012,   102,  2002,
          4122,  2000,  2022,  2019,  9932,  3992,   999,   102,     0,     0,
             0],
        [  101,  3958, 27227,  2001,  1037, 25732,  3992,  1012,   102,  2002,
          4122,  2000,  2022,  2019,  9932,  3992,  1012,   102,     0,     0,
             0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

**Decreasing from positive tone to negative tone**

## 3. Optimize inference

### How Inference Works

#### Encoder-Decoder 
- **Prefill**: The encoder processes all input tokens in parallel, producing K,V representations — essentially a rich context bank.
- **Decode**: The decoder cross-attends to those K,V representations and generates output tokens one at a time. The **first output token is generated in the first decode step**, not during prefill.

#### Decoder-Only 
- **Prefill**: All prompt tokens are processed in **parallel** in a single forward pass. This populates the KV cache for all prompt positions and computes logits at the final position — the **first output token is sampled here**, at the end of prefill.
- **Decode**: After prefill, the model generates **one token at a time**, attending to all previous tokens via the KV cache (no recomputation needed). This phase is sequential and bottlenecked by memory bandwidth — why long outputs are slower than long prompts.

---
#### KV Cache
- During Inference only
- For Decoder only model
- Self attention layer
- When generating text one at a time, K, V are matrices that store the key and value vectors for each token position. When adding a new token, the model only needs to compute the K,V for that token and can reuse the cached K,V for all previous tokens. 
- Why not do Q cache? Because Q is only needed for the current token, so it is a vector. 
- Any trade-off? KV cache uses more VRAM. Memory for speed. 
- More in [HF KV Cache](https://huggingface.co/docs/transformers/en/kv_cache)

---

## Inference Parameters

- **`max_new_tokens`** — Maximum number of tokens to generate. Generation stops at this limit even if no stop condition is hit.

- **`temperature`** — Controls randomness by scaling logits before sampling.
  - `0.0` → Deterministic, always picks the highest probability token
  - `0.7` → Moderately creative, still coherent (common default)
  - `1.0` → Sampling at the true model distribution
  - `>1.0` → More random, less coherent

- **`top_p`** — Nucleus sampling. At each step, only consider tokens whose cumulative probability sums to `top_p`, then sample from that subset.
  - `1.0` → Consider all tokens (no filtering)
  - `0.95` → Use tokens covering 95% of probability mass (common default)
  - `0.5` → Very conservative, only high-confidence tokens
  - Note: `temperature` reshapes the distribution first, then `top_p` filters it.

- **`do_sample`** — If `False`, uses greedy decoding (always picks top token). If `True`, enables sampling with `temperature`/`top_p`.

- **`stop_sequences`** — List of strings that immediately halt generation if produced. Empty list means no custom stop — generation runs until `max_new_tokens` or the model EOS token.

- **`details`** — Returns extra metadata alongside the generated text: token-level log probabilities, finish reason (`length`, `eos_token`, `stop_sequence`), and per-token details. Useful for debugging.

---

### Optimization Techniques
```
GPU Memory Types
  GPU
  ├── Compute  →  tensor cores (does the math)
  └── Memory
      ├── VRAM (HBM — High Bandwidth Memory) — the main GPU memory. Stores model weights, KV cache, activations.   →  large but slower  (~40-80GB on A100)
      └── SRAM — on-chip cache, This is where the GPU does active computation. →  tiny but fast     (~40MB on A100)


  The wire between VRAM and SRAM has limited bandwidth — this is the root cause of most LLM inference bottlenecks.
```
Each technique targets a different bottleneck in the GPU:

| Technique | What it Fixes | How |
|---|---|---|
| **Flash Attention** | Compute efficiency | Reduces redundant VRAM↔SRAM transfers during attention by tiling computation in fast SRAM |
| **Paged Attention** | Memory efficiency | Eliminates wasted KV cache space and fragmentation by dividing KV cache into small, non-contiguous "pages"s (used in vLLM) |
| **Quantization** | Memory capacity | Shrinks model weights from fp32/fp16 to int8/int4, making models fit in VRAM with minimal quality loss |

```
  Standard attention — multiple round trips:                                                                                                                                    
  VRAM → load Q, K      → SRAM → compute QKᵀ scores    → write back to VRAM                                                                                                     
  VRAM → reload scores  → SRAM → compute softmax        → write back to VRAM                                                                                                    
  VRAM → reload scores  → SRAM → multiply by V          → write back to VRAM                                                                                                    
  Each intermediate result (the score matrix) gets written to VRAM and reloaded — because SRAM is too small to hold it all at once.                                             
                                                                                                                                                                                
  Flash Attention — tiling:                                                                                                                                                     
  SRAM → load small block of Q, K, V → compute partial attention →
         accumulate result → load next block → ...                                                                                                                              
  It processes the attention in small tiles that fit entirely in SRAM, keeping everything on-chip and updating a running result. Never writes the full score matrix to VRAM at  
  all.                                                                                                                                                                          
                                                                                                                                                                                
  So what actually changes:                                                                                                                                                     
  - The math: identical                                                                                                                                                         
  - The memory access pattern: fewer round trips to VRAM
  - The bottleneck removed: memory bandwidth (the wire between VRAM and SRAM), not compute
```

#### Mixed Precision & Quantization 

**Precision formats:**

| Format | Sign | Exponent | Mantissa | Characteristics |
|---|---|---|---|---|
| fp32 | 1 | 8 | 23 | Range: huge, precision: high |
| fp16 | 1 | 5 | 10 | Range: small, precision: medium |
| bf16 | 1 | 8 | 7 | Range: huge, precision: lower |

bf16 allocate more bits to the exponent than fp16, which helps maintain a similar range as fp32. This means:
- No underflow/overflow problems (same range as fp32)
- Half the memory of fp32
- Fast on modern GPU tensor cores

- `bf16` — preferred for **training** (same range as fp32, half the memory). Range here means the exponent range, so it can represent very large and very small numbers without overflow/underflow issues.

- `fp16` — preferred for **inference** (fast, fits more in VRAM). Has a smaller exponent range than bf16, so it can underflow to zero for very small values or overflow to infinity for very large values — but this is usually not a problem for inference. On the other hand, fp16 can be faster on consumer GPUs that don't support bf16.

- `int8/int4` — **quantized inference** (4× smaller, slight quality loss; enables running on consumer hardware)


**Apply mixed precision**
Using fp16 as an example, Mixed precision uses fp16 for speed and memory savings where it's safe, and fp32 where numerical accuracy is critical — giving you faster training, lower memory usage, and nearly identical results compared to pure fp32.

| Component | Precision | Reason |
|---|---|---|
| Model weights | fp16 | Stored small, saves memory |
| Gradient computation | fp32 | Needs precision, avoids errors |
| Activations | fp16 | Fast computation |
| Loss scaling | fp32 | Critical, must be accurate |


```
Training (Mixed Precision):
  Weights stored in FP16/BF16   ← saves memory
  Forward pass in FP16/BF16     ← fast tensor core math
  Gradients computed in fp32    ← prevents underflow
  Master weights in fp32        ← accurate weight updates

Inference (convert the model to lower precision to maximize throughput):
  Weights loaded in fp16        ← fits more model in VRAM
  Computation in fp16           ← faster than fp32
  (no gradients needed at all)  ← simpler than training
```

**Quantization: compress the model with lower precision. (int8/int4)**
- Default model download: fp32 → too big for most GPUs
- fp16 model: half the size, nearly identical output quality
- int4 quantization: 4× smaller, slight quality loss

fp16 is the standard "I want good quality but fit in my GPU" choice. int4 is "I want to run on a laptop."


**Mixed Precision vs Quantization: Key Differences. Is It Reversible?**

- Mixed precision is a training/compute strategy — switching between precisions during the math to go faster while staying accurate.
- Quantization is a compression strategy — permanently shrinking stored weights to save memory, accepting a small quality tradeoff.


The Key Dimension: 
This is the deepest difference.
Mixed precision:   fp16 ←→ fp32   (you switch back and forth, model preserves the original fp32 weights, so no information is lost)
Quantization:      fp32  →  int8/int4  (one way, information is permanently lost)

In mixed precision, the fp32 "master copy" of weights always exists. You're just doing the math in fp16 temporarily for speed, then updating the precise fp32 weights.
In quantization, you throw away the original fp32 weights and replace them with int4. You can't fully recover the original values. It's a lossy compression — like saving a JPEG instead of a RAW photo.

#### System Prompts and Instructions

In [17]:
# system prompt is not visible to the user, but it can be used to set the behavior of the model. For example, you can use a system prompt to tell the model to be more creative or to focus on a specific topic. The system prompt is typically included in the input text that is fed into the model, but it is not shown to the user in the output.
from transformers import AutoModelForCausalLM, AutoTokenizer
checkpoint = "HuggingFaceTB/SmolLM2-360M-Instruct"

device = "cuda" # for GPU usage or "cpu" for CPU usage
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# for multiple GPUs install accelerate and do `model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto")`
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

messages = [{"role": "user", "content": "What is the capital of France."}]
input_text = tokenizer.apply_chat_template(messages, tokenize=False)
print(input_text)
encoded = tokenizer(input_text, return_tensors="pt").to(device)
outputs = model.generate(
    encoded["input_ids"],
    attention_mask=encoded["attention_mask"],
    max_new_tokens=50,
    temperature=0.2,
    top_p=0.9,
    do_sample=True
)
print(tokenizer.decode(outputs[0]))


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of France.<|im_end|>

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of France.<|im_end|>
<|im_start|>assistant
The capital of France is Paris.<|im_end|>


# 4. Fine Tune



**Overall flow:**
```
  Pretraining (self-supervised, massive data)    we can download pre-trained model                                    
        ↓                 
  Supervised Fine-Tuning  ← for specific task with specific data. can change model structure by adding a (final) layer
        ↓                
  RL Post training ← further align. discuss next sec
```

- SFT (Supervised Fine Tuning) — standard fine-tuning on a labeled dataset for a specific task (e.g. sentiment analysis, NER, etc.)
- RLPT (Reinforcement Learning Post Training) — fine-tuning a pre-trained model using reinforcement learning signals (e.g. human feedback, reward models) to further align it with desired behaviors or values.


**Task**

Pre-trained model:

"bert-base-uncased" is a pre-trained model hosted on the Hugging Face Hub. 

- The transformer encoder weights (BERT's body) are pre-trained. When you call ```AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")```, it downloads the weights that were pre-trained by Google on large text corpora (BooksCorpus + English Wikipedia) using masked language modeling.     
- The classification head (a linear layer on top) is randomly initialized, because BERT was not originally trained for sequence classification

Data:

It's a dataset of sentence pairs, where each pair is labeled:
- 1 — the two sentences are paraphrases (same meaning)
- 0 — they are not paraphrases

| Sentence 1                  | Sentence 2                        | Label |
|-----------------------------|-----------------------------------|-------|
| "He said the food was good." | "He mentioned the meal was tasty." | 1     |
| "The stock rose 2%."         | "The company lost $3M."            | 0     |

**Which part of the model is pre-trained?**
- The transformer encoder layers (the "body" of BERT) are pre-trained on masked language modeling.
- The classification head (a linear layer on top) is randomly initialized, because BERT was not originally trained for sequence classification. During fine-tuning, the encoder layers are updated from their pre-trained state, and the classification head learns to map the encoder's output to the paraphrase labels.

When we fine-tune, we update both the pre-trained encoder and the randomly initialized classification head. The encoder adapts its representations to better suit the paraphrase detection task, while the classification head learns to interpret those representations to predict the correct labels. 

**Do the original weights matter?**

Absolutely — this is the whole point of transfer learning. BERT's pretrained weights encode rich language understanding (grammar, semantics, context) learned from billions of tokens. Fine-tuning nudges those weights slightly toward the task, rather than learning from scratch. Without them you'd need vastly more data.

**Risk of overfitting:**
bert-base-uncased has 110M parameters:
| Component | Parameters |
|---|---|
| Embedding layer | ~23M |
| 12 Transformer layers | ~85M |
| New classifier head (weight + bias) | ~1.5K |
| Total | ~110M |
With only a few thousand MRPC samples, updating all 110M parameters risks overfitting.

Option - freezing the lower layers of the model is a safer approach to prevent overfitting when you have limited data. This way, you only update the top layers and the new classification head, which reduces the risk of overfitting while still allowing the model to adapt to the new task.

**LoRA (Low-Rank Adaptation):** A parameter-efficient fine-tuning technique that freezes the original model weights and introduces trainable low-rank matrices to approximate the weight updates. This reduces the number of trainable parameters significantly while maintaining performance.

**Basic Logic:**
- Freeze the pre-trained weights $W$ of the model.
- Introduce two low-rank matrices $A$ (shape $d \times r$) and $B$ (shape $r \times k$), where $r \ll \min(d, k)$ is the rank (typically small, e.g., 8 or 16).
- The effective weight update is $\Delta W = A \cdot B^T$, added to the original $W$.
- During forward pass: $W' = W + \Delta W = W + A \cdot B^T$.
- Only $A$ and $B$ are trained, keeping the original $W$ unchanged.

This allows fine-tuning with far fewer parameters (e.g., 0.5-1% of original), enabling faster training and lower memory usage. LoRA is especially useful for large models like BERT or GPT, where full fine-tuning is expensive.
 

Some other examples for fine tune in HF:[Fine tune a causal language model](https://huggingface.co/docs/transformers/tasks/language_modeling#causal-language-modeling) 


### Understand Fine-Tuning Loop


```
forward → backward → optimizer step → scheduler step → zero gradients
```

- Freeze model parameters: `requires_grad=False`
- Change layer behavior (disable dropout and batch normalization): `model.eval()` vs `model.train()`
- Disable gradient computation during evaluation: `torch.no_grad()`

**Accelerator:** wrap key objects with `accelerator.prepare()` and use `accelerator.backward()` instead of `loss.backward()`


**Learning Curve of accuracy and loss for classification:** 
- loss can be more smooth. The loss can improve if the model's output gets closer to the target, even if the final prediction is still incorrect. 
- accuracy can be fluctuate. Accuracy only improves when the prediction crosses the threshold to be correct.

In [3]:

# Load model and tokenizer from the checkpoint
checkpoint = "bert-base-uncased"
# model and tokenizer should come from the same checkpoint
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
# show data
sequences = [
    "I've been waiting for a HuggingFace course my whole life. And it is what I want",
    "This course is amazing!",
]
batch = tokenizer(sequences[0],sequences[1], padding=True, truncation=True, return_tensors="pt")
print(batch) 
batch["labels"] = torch.tensor([1, 1])
print(batch.labels) 


{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,  1998,  2009,  2003,  2054,  1045,
          2215,   102,  2023,  2607,  2003,  6429,   999,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
         1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1]])}
tensor([1, 1])


In [5]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# UNEXPECTED — weights that exist in the checkpoint (bert-base-uncased) but are not used by BertForSequenceClassification:                                                           
#  - These are the MLM (Masked Language Modeling) and NSP (Next Sentence Prediction) heads that BERT was originally pretrained with — they're simply discarded.
# swapping BERT's pretraining heads for a new classification head. This is expected and intentional — it's the whole point of fine-tuning. 
# The missing weights will be learned during your training loop.     


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
from datasets import load_dataset

raw_datasets = load_dataset("glue", "mrpc")
print("dataset key", raw_datasets.keys()    )  # → dict_keys(['train', 'validation', 'test'])
raw_train_dataset = raw_datasets["train"]
print("data example:")
raw_train_dataset[0]
print("feature keys:", raw_train_dataset.features.keys())  # → dict_keys(['idx', 'label', 'sentence1', 'sentence2'])

README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

dataset key dict_keys(['train', 'validation', 'test'])
data example:
feature keys: dict_keys(['sentence1', 'sentence2', 'label', 'idx'])


In [ ]:
raw_train_dataset[10]

{'sentence1': 'Legislation making it harder for consumers to erase their debts in bankruptcy court won overwhelming House approval in March .',
 'sentence2': 'Legislation making it harder for consumers to erase their debts in bankruptcy court won speedy , House approval in March and was endorsed by the White House .',
 'label': 0,
 'idx': 11}

In [23]:
raw_train_dataset[15]

{'sentence1': 'Rudder was most recently senior vice president for the Developer & Platform Evangelism Business .',
 'sentence2': 'Senior Vice President Eric Rudder , formerly head of the Developer and Platform Evangelism unit , will lead the new entity .',
 'label': 0,
 'idx': 16}

In [7]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

In [7]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)
    """
    truncation=True, max_length=128 as default

    When you pass two texts (e.g. sentence pairs for MRPC), the tokenizer automatically formats them as:
    [CLS] sentence_A [SEP] sentence_B [SEP]
    You don't need to concatenate manually. The tokenizer handles it and also sets token_type_ids to distinguish which tokens belong to sentence A (0) vs sentence B (1) — which BERT
    uses to understand it's a pair.
    """
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Remove string columns that cannot be collated into tensors
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets


Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [14]:

# Remove string columns that cannot be collated into tensors
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [ ]:
# collate to handle batch level processing
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
print("Check input lengths:", [len(x) for x in samples["input_ids"]])

batch = data_collator(samples)
print("Batch shapes:", {k: v.shape for k, v in batch.items()})

print("after data collator, each sequence has been padded to the same length, and the batch is ready to be fed into the model.")

Check input lengths: [50, 59, 47, 67, 59, 50, 62, 32]
Batch shapes: {'labels': torch.Size([8]), 'input_ids': torch.Size([8, 67]), 'token_type_ids': torch.Size([8, 67]), 'attention_mask': torch.Size([8, 67])}
after data collator, each sequence has been padded to the same length, and the batch is ready to be fed into the model.


In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer", eval_strategy="epoch")


In [18]:
tokenized_datasets["validation"].shape

(408, 7)

In [10]:
print("validation data shape:", tokenized_datasets["validation"].shape)
print("training data shape:", tokenized_datasets["train"].shape)
print("test data shape:", tokenized_datasets["test"].shape)


validation data shape: (408, 7)
training data shape: (3668, 7)
test data shape: (1725, 7)


In [10]:
# "test-trainer" is the directory where the model predictions and checkpoints will be saved. You can choose any name you want for this directory.
# eval_strategy="epoch" means that the evaluation will be performed at the end of each epoch during training. This allows you to monitor the model's performance on the validation set after each epoch and make decisions based on those results, such as early stopping or saving the best model checkpoint.
 

from transformers import Trainer

def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments("test-trainer", eval_strategy="epoch") 

trainer = Trainer(
    model,
    training_args, 
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)
preds = np.argmax(predictions.predictions, axis=-1)

(408, 2) (408,)


In [33]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.573634,0.813725,0.875817
2,0.340914,0.600824,0.850490,0.895726
3,0.231251,0.771883,0.850490,0.897133


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=0.23904371296225377, metrics={'train_runtime': 210.2045, 'train_samples_per_second': 52.349, 'train_steps_per_second': 6.551, 'total_flos': 405114969714960.0, 'train_loss': 0.23904371296225377, 'epoch': 3.0})

#### Accelerater

In [17]:
 
# ["attention_mask", "input_ids", "labels", "token_type_ids"]

from torch.utils.data import DataLoader
 

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print("training steps:", num_training_steps)

import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("check if GPU is available:", torch.cuda.is_available(),"device is", device)

model.to(device)


training steps: 1377
check if GPU is available: True device is cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [18]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

  0%|          | 0/1377 [00:00<?, ?it/s]

**Full accelerated training loop with Trainer API**

In [21]:
from accelerate import Accelerator
from transformers import AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW
from tqdm.auto import tqdm
import evaluate
import torch

def training_function():
    accelerator = Accelerator()

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
    optimizer = AdamW(model.parameters(), lr=3e-5)

    train_dl, eval_dl, model, optimizer = accelerator.prepare(
        train_dataloader, eval_dataloader, model, optimizer
    )

    num_epochs = 3
    num_training_steps = num_epochs * len(train_dl)
    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )

    progress_bar = tqdm(range(num_training_steps))
    metric = evaluate.load("glue", "mrpc")

    for epoch in range(num_epochs):
        # Training
        model.train()
        for batch in train_dl:
            outputs = model(**batch)
            loss = outputs.loss
            accelerator.backward(loss)

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)

        # Evaluation
        model.eval()
        for batch in eval_dl:
            with torch.no_grad():
                outputs = model(**batch)
            predictions = outputs.logits.argmax(dim=-1)
            predictions, references = accelerator.gather_for_metrics(
                (predictions, batch["labels"])
            )
            metric.add_batch(predictions=predictions, references=references)

        results = metric.compute()
        print(f"Epoch {epoch + 1}: {results}")


In [22]:
from accelerate import notebook_launcher

notebook_launcher(training_function, num_processes=1)


Launching training on one GPU.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/1377 [00:00<?, ?it/s]

Epoch 1: {'accuracy': 0.8504901960784313, 'f1': 0.8884826325411335}
Epoch 2: {'accuracy': 0.8823529411764706, 'f1': 0.9180887372013652}
Epoch 3: {'accuracy': 0.8823529411764706, 'f1': 0.9172413793103448}



---




### Advanced Optimization Techniques


In [48]:


%pip install -qqq torch torchvision setuptools scikit-learn

# Install Hugging Face libraries
%pip install  --upgrade datasets -qqq accelerate hf-transfer transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 88.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.9 MB/s eta 0:00:00:00:0100:01


In [ ]:
from datasets import load_dataset

# Dataset id from huggingface.co/dataset
dataset_id = "burtenshaw/PleIAs_common_corpus_code_classification"

# Load raw dataset
dataset = load_dataset(dataset_id)

In [50]:
print(len(dataset["train"]))
print(dataset["train"][0])

127723
{'text': '/*\n * Copyright (c) 2000 Kungliga Tekniska Högskolan\n * (Royal Institute of Technology, Stockholm, Sweden).\n * All rights reserved.\n *\n * Redistribution and use in source and binary forms, with or without\n * modification, are permitted provided that the following conditions\n * are met:\n *\n * 1. Redistributions of source code must retain the above copyright\n *    notice, this list of conditions and the following disclaimer.\n *\n * 2. Redistributions in binary form must reproduce the above copyright\n *    notice, this list of conditions and the following disclaimer in the\n *    documentation and/or other materials provided with the distribution.\n *\n * 3. Neither the name of the Institute nor the names of its contributors\n *    may be used to endorse or promote products derived from this software\n *    without specific prior written permission.\n *\n * THIS SOFTWARE IS PROVIDED BY THE INSTITUTE AND CONTRIBUTORS ``AS IS\'\' AND\n * ANY EXPRESS OR IMPLIED W

In [ ]:
from transformers import AutoTokenizer

# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-base"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Tokenize helper function
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, return_tensors="pt")

# Tokenize dataset
tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])

tokenized_dataset["train"].features.keys()
# dict_keys(['labels', 'input_ids', 'attention_mask'])

In [53]:
tokenized_dataset.shape

{'train': (127723, 3), 'test': (14192, 3)}

In [55]:
tokenized_dataset['train'].features.keys()

dict_keys(['labels', 'input_ids', 'attention_mask'])

In [58]:
from transformers import AutoModelForSequenceClassification

# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-base"

# Prepare model labels - useful for inference
labels = list(set(tokenized_dataset["train"]["labels"]))
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label



In [68]:
tokenized_datasets.save_to_disk("tokenized_datasets")

Saving the dataset (0/1 shards):   0%|          | 0/3668 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/408 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1725 [00:00<?, ? examples/s]

In [ ]:
  !pip install -U transformers

In [ ]:
  !pip install -U huggingface_hub   

In [63]:
import transformers   
print(transformers.__version__)  
from transformers import AutoModelForSequenceClassification

5.0.0


In [67]:
import importlib                                                                                                                                                                   
import transformers
importlib.reload(transformers)     

<module 'transformers' from '/usr/local/lib/python3.12/dist-packages/transformers/__init__.py'>

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

In [ ]:
 

import numpy as np
from sklearn.metrics import f1_score

# Metric helper method
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    score = f1_score(
            labels, predictions, labels=labels, pos_label=1, average="weighted"
        )
    return {"f1": float(score) if score == 1 else score}

from huggingface_hub import HfFolder
from transformers import Trainer, TrainingArguments

# Define training args
training_args = TrainingArguments(
    output_dir= "ModernBERT-code-classifier",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    num_train_epochs=5,
    bf16=True, # bfloat16 training
    optim="adamw_torch_fused", # improved optimizer
    # logging & evaluation strategies
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="epoch",  # conduct evaluation every epoch
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    # push to hub parameters
    push_to_hub=True,
    hub_strategy="every_save",
    hub_token=HfFolder.get_token(),
    report_to="wandb" # log the result to a platform Weight & Biasis.
)



In [ ]:
limited_dataset = tokenized_dataset["train"].select(range(100))

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=limited_dataset,
    eval_dataset=tokenized_dataset["test"],  # if no evaluation data, the evaluation result will not be posted.
    compute_metrics=compute_metrics,
)
trainer.train()

# Context Length and attention span

- **Definition**: Maximum sequence length a model can process
    - BERT: 512 tokens
    - Larger models: 1024+ tokens
    - Information beyond this limit is truncated and not considered

- **In Chat-based LLMs**: Context length determines conversation history scope
    - Earlier messages truncated when conversation exceeds limit
    - Can cause loss of important information
    - Results in less coherent responses

- **In Agent Building**: Context length limits memory and tool usage history
    - May forget important past actions if interactions exceed limit
    - Affects decision-making quality
    - Leads to suboptimal performance

How agent manage context length?
- **Summarization**: Condense past interactions into a summary to fit within context
- **Retrieval-Augmented Generation**: Retrieve relevant past information on demand instead of keeping everything in context
- **Hierarchical Memory**: Use a structured memory system to store long-term information outside of the immediate context, and retrieve it as needed

## RL Post Training

Increase the probability of action that generate higher expected outcome.
 
### What It Is
- **RL Post Training** (also called RLHF — Reinforcement Learning from Human Feedback) is a training stage applied *after* supervised fine-tuning (SFT) to align model behavior with human preferences
- The model is treated as a **policy**: it takes a state (prompt + context) and produces an action (the next token / response)
- A **reward model** scores outputs based on a defined signal that evaluates output quality can serve as the reward functio; the LLM is updated to maximize that reward

### SFT vs RL Post Training — Mental Model

**SFT** — "here is the correct output, copy it"
- Label = the exact target sequence
- Loss = how far your output is from that target
- Model has no freedom to explore

**RL Post Training** — "here is how good your output was, adjust accordingly"
- Label = a scalar reward score (not the output itself)
- Loss = shaped by reward relative to your own other attempts
- Model explores by sampling, then learns from contrast

### Key Stages
- **Reward Modeling** — train a separate model on human preference pairs (response A vs B) to predict which output humans prefer
- **RL Optimization** — use the reward model as a signal to update the LLM policy (e.g. via PPO or GRPO)
- **KL Penalty** — add KL divergence to model rewrd to prevent the model from drifting too far from the SFT baseline (reward hacking)

### Optimization Strategy
- **PPO** (Proximal Policy Optimization) — classic RLHF approach; stable but computationally expensive
- **DPO** (Direct Preference Optimization) — skips the separate reward model; directly trains on preference pairs, simpler and more efficient
- **GRPO** (Group Relative Policy Optimization) — samples multiple outputs, ranks them, trains on relative rewards without a critic model

### Connection to AI Agents
- **Agents act sequentially** — RL is a natural fit because agent decisions are a chain of actions over time, exactly what RL optimizes for
- **Reward = task success** — instead of human preference scores, agent rewards come from environment feedback (did the tool call succeed? was the task completed?)
- **Long-horizon reasoning** — RL post training teaches models to plan ahead across multiple steps, which is essential for agents that use tools, browse the web, or write/run code
- **Self-improvement loop** — agents can generate their own training data by attempting tasks, scoring outcomes, and learning from both successes and failures (e.g. AlphaCode, OpenAI o1)
- **Alignment in agentic contexts** — RL helps enforce safe behavior (e.g. not taking irreversible actions) by shaping the reward signal around safety constraints

---

### Mapping to RL Concepts

| RL Concept | LLM Context |
|---|---|
| Agent | The LLM |
| State | The prompt / conversation so far |
| Action | The generated response (or each token) |
| Policy | The model's probability distribution over outputs |
| Reward | Human preference score / accuracy / format check |
| Update rule | Increase probability of high-reward outputs |

### Who Defines the Reward

"Human preference" is one source of reward, but the RL framework is more general — any signal that evaluates output quality can serve as the reward function:

- **RLHF** — humans rank outputs → reward model learns human preference → scores future outputs
- **GRPO** — reward is a deterministic function (format check, answer correctness) — no human raters or separate reward model needed
- **Agentic RL** — reward comes from the environment (did the code run? did the API call succeed?)


---

### GRPO in Practice (from HF Cookbook)

Reference: [Fine-tuning VLM with GRPO and TRL](https://huggingface.co/learn/cookbook/fine_tuning_vlm_grpo_trl)

**Step 1 — Data Preparation**
- Use datasets with reasoning traces (e.g. `lmms-lab/multimodal-open-r1-8k-verified`)
- Each example contains: image + problem, solution, and a thinking/reasoning trace
- The reasoning trace teaches the model *how* to think, not just *what* to answer

**Step 2 — Model Setup**
- Start from an instruction-tuned base (e.g. `Qwen2.5-VL-3B-Instruct`)
- Use TRL's `GRPOTrainer` which handles the RL loop internally

**Step 3 — Reward Functions**
- No single reward model — instead define multiple reward functions:
  - **Solution correctness** — is the final answer right?
  - **Reasoning quality** — are the intermediate steps logical and complete?
- Rewards are computed **relative to a group** of sampled outputs for the same prompt (not an absolute score)

**Step 4 — GRPO vs PPO**

| | PPO | GRPO |
|---|---|---|
| Advantage baseline | Value/critic network | Group mean reward |
| Critic model needed | Yes | No |
| Memory | Higher | More efficient |
| Best for | General RL | Reasoning / ranking tasks |

GRPO samples multiple responses per prompt, computes each response's reward, then trains the model to favor responses that scored above the group average — no separate critic model needed.

**Step 5 — Training Loop**
```python
from trl import GRPOTrainer, GRPOConfig

grpo_config = GRPOConfig(
    output_dir="./grpo_model",
    num_train_epochs=3,
    learning_rate=1e-5,
    per_device_train_batch_size=4,
)

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=train_dataset,
    reward_functions=[reasoning_quality_reward, solution_correctness_reward],
    processing_class=processor,
)

trainer.train()
```